**NAME**: Sumit Bansiya

**ROLL NO.**: 25901337

**CP LAB PRACTICALS**


### Question 1: Easy (CO2) – Extracting Diagonal Elements using CUDA

**Objective:** To extract the main diagonal elements from our 1024 x 1024 matrix (DS2 dataset) using parallel computation on the GPU.

**How the Algorithm Works:**

* **The Dataset:** We are working with a square matrix. The "main diagonal" consists of elements where the row index and the column index are exactly the same (for example, row 0, column 0; row 1, column 1; and so on).
* **GPU Threading (The Parallel Advantage):** If we were using a standard CPU, we would write a `for` loop to go through the matrix one step at a time. With CUDA, we skip the loop entirely. Instead, we assign a single GPU thread to handle each element on the diagonal simultaneously. Since there are 1024 elements on the diagonal, we set up a 1D grid of threads.
* **Execution Steps:** 1. Each thread calculates its own unique global ID (its position in the line of threads).
2. The thread performs a quick check to make sure its ID isn't larger than our matrix size, which prevents out-of-bounds memory errors.
3. Finally, each thread directly copies the value at `matrix[ID, ID]` into a new 1D result array.

**Why this is efficient:** This method drastically reduces the execution time. Because all 1024 elements are fetched at the exact same time by different GPU threads, we turn a sequential task into a lightning-fast parallel one.

In [ ]:
import numpy as np
from numba import cuda
import math
import matplotlib.pyplot as plt

# Generate the datasets as specified
# DS2: 1024 x 1024 matrix
DS2_N = 1024
ds2_matrix = np.random.rand(DS2_N, DS2_N).astype(np.float32)

# DS3: 2048 x 2048 matrix
DS3_N = 2048
ds3_matrix = np.zeros((DS3_N, DS3_N), dtype=np.float32)

print(f"DS2 Matrix Shape: {ds2_matrix.shape}")
print(f"DS3 Matrix Shape: {ds3_matrix.shape}")

DS2 Matrix Shape: (1024, 1024)
DS3 Matrix Shape: (2048, 2048)


In [ ]:
# --- CUDA Kernel Definition ---
@cuda.jit
def extract_diagonal_kernel(matrix, diagonal, N):
    # Calculate the 1D thread index
    idx = cuda.grid(1)

    # Check bounds to avoid segmentation faults
    if idx < N:
        diagonal[idx] = matrix[idx, idx]

# --- Host Code Execution ---
# 1. Allocate device memory and copy data to GPU
d_matrix = cuda.to_device(ds2_matrix)
d_diagonal = cuda.device_array(DS2_N, dtype=np.float32)

# 2. Configure thread blocks and grid
threads_per_block = 256
blocks_per_grid = (DS2_N + (threads_per_block - 1)) // threads_per_block

# 3. Launch Kernel
extract_diagonal_kernel[blocks_per_grid, threads_per_block](d_matrix, d_diagonal, DS2_N)

# 4. Copy result back to host (CPU)
result_diagonal = d_diagonal.copy_to_host()

print("First 10 Diagonal Elements:")
print(result_diagonal[:10])

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


First 10 Diagonal Elements:
[0.9273052  0.1661286  0.93220294 0.79759854 0.02801952 0.15179011
 0.06678504 0.7431314  0.08049203 0.9630192 ]
